# OpenAI API 기초 — Chat Completions · Responses(웹 검색) · 스트리밍 · 토큰과 요금

수업 필기 `openai_api_실습` 을 실행 가능하도록 정리 · 개선한 노트입니다. (강의 코드 `openai_api_강의본` 과 대조)

- 📖 교재 : `AI Agent` 교재10 **p.2~11** (22. 생성형 AI의 언어 모델(LLM) — API · JSON · OpenAI API 사용법 · 토큰 · API 키 관리)
- 🔗 이어지는 노트 : [`prompt_output_format.ipynb`](./prompt_output_format.ipynb) (프롬프트 · 출력 형식) → [`review_sentiment_analysis.ipynb`](./review_sentiment_analysis.ipynb) (감성 분석)
- 📝 원본에서 고친 곳은 그 자리에 **📝 고친 부분** 으로 적고 맨 끝 표에 모았습니다. 그림은 [`images/`](./images/) 폴더

> **이 노트의 출력을 읽는 법** — OpenAI 서버를 부르는 셀은 API 키(유료)가 있어야 돌아갑니다. 정리하는 환경에는 키가 없어서
> - `# 📼` 로 시작하는 셀 : **수업 시간에 실제로 실행한 결과**(필기 원본의 출력)를 그대로 옮겼습니다.
> - `# ⏳` 로 시작하는 셀 : 수업 때 결과가 남지 않은 셀이라 **출력이 비어 있습니다.** `.env` 를 준비하고 직접 실행하세요.
> - 표시가 없는 셀 : 이 노트에서 실제로 실행한 결과입니다 (토큰 세기 · JSON 변환 · 오류 재현 · 사용량 계산).

## 0. 준비 (교재 p.10)

1. **설치** : `pip install openai python-dotenv tiktoken`
2. **API 키** : [platform.openai.com](https://platform.openai.com/api-keys) 에서 발급(결제 수단 등록 필요) → 노트북과 같은 폴더(또는 상위 폴더)에 `.env` 파일을 만들고 한 줄을 적는다

```
OPENAI_API_KEY="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
```

3. **`.env` 는 절대 GitHub 에 올리지 않는다** — 이 저장소의 `.gitignore` 에 `.env` 가 들어 있는지 확인(들어 있음). 키가 노출되면 다른 사람이 내 요금으로 쓸 수 있다
4. 맨 위부터 **모두 실행**. 호출 한 번에 몇 초~수십 초가 걸리고 요금이 나간다(3절에서 계산)

In [1]:
# 필요한 함수 / 클래스 임포트
import os, json
from dotenv import load_dotenv, find_dotenv   # .env 파일 → 환경 변수
from openai import OpenAI                     # OpenAI 서버와 통신하는 클라이언트

## 1. API 와 JSON (교재 p.3~5)

- **API** : 프로그램끼리 대화하는 규칙. 내 파이썬 코드(클라이언트)가 **요청**을 보내면 OpenAI 서버가 **응답**을 돌려준다
- 주고받는 데이터의 모양이 **JSON** — 키-값 쌍으로 된 **문자열**

| | Python `dict` | JSON 문자열 |
|---|---|---|
| 자료형 | `dict` (메모리 안의 객체) | `str` (글자) → 네트워크로 보낼 수 있다 |
| 따옴표 | `'` · `"` 모두 | **`"` 만** |
| 참 / 거짓 / 없음 | `True` · `False` · `None` | `true` · `false` · `null` |
| 바꾸기 | `json.dumps(dict)` → 문자열 | `json.loads(문자열)` → dict |

![요청 · 응답 애니메이션](./images/anim_01_request_response.gif)

In [2]:
# dict ↔ JSON 문자열 변환
book = {'book_id': 'B001', 'title': '달빛 아래의 서약', 'in_stock': True, 'series': None}

book_json = json.dumps(book, ensure_ascii=False)   # dict → JSON 문자열 (ensure_ascii=False : 한글을 그대로)
print(type(book).__name__, ':', book)
print(type(book_json).__name__, ' :', book_json)    # 큰따옴표, true, null 로 바뀐다

book_back = json.loads(book_json)                   # JSON 문자열 → dict
print('되돌린 값이 원래 dict 와 같은가 :', book_back == book)

dict : {'book_id': 'B001', 'title': '달빛 아래의 서약', 'in_stock': True, 'series': None}
str  : {"book_id": "B001", "title": "달빛 아래의 서약", "in_stock": true, "series": null}
되돌린 값이 원래 dict 와 같은가 : True


In [3]:
# dict 를 그냥 str() 로 바꾼 글자는 JSON 이 아니다 (작은따옴표 · True · None)
not_json = str(book)
print(not_json)
try:
    json.loads(not_json)
except json.JSONDecodeError as e:
    print('JSONDecodeError :', e)

{'book_id': 'B001', 'title': '달빛 아래의 서약', 'in_stock': True, 'series': None}
JSONDecodeError : Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


`str(dict)` 는 작은따옴표라서 `json.loads` 가 첫 글자 다음에서 바로 실패한다. 프롬프트에 **JSON 예시**를 넣을 때도 `json.dumps` 로 만든 문자열을 써야 한다 (감성 분석 노트에서 다시 나온다).

## 2. 환경 설정 — API 키 읽기 · 클라이언트 만들기 (교재 p.6~7 · p.10)

In [4]:
# .env 파일 → 환경 변수 OPENAI_API_KEY
env_path = find_dotenv(usecwd=True)      # 지금 폴더에서 위로 올라가며 .env 를 찾는다 (없으면 '')
loaded = load_dotenv(env_path)           # 읽었으면 True
print('.env 읽음 :', loaded)
print('OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))   # 키 값 자체는 절대 print 하지 않는다

.env 읽음 : True
OPENAI_API_KEY 있음 : True


> **📝 고친 부분** — 필기는 `key_path = "C:\\Users\\Jinho\\work\\.env"` 처럼 **내 PC 의 절대 경로**를 적었다. 다른 PC · Colab · 팀원 환경에서는 그 경로가 없어서 `load_dotenv` 가 `False` 를 내고, 뒤의 `OpenAI()` 에서 키가 없다는 오류가 난다. `find_dotenv()` 로 찾게 하고, 키가 **있는지만** 확인한다.

In [5]:
# OpenAI 서버와의 연결 통로(클라이언트) 생성 — 환경 변수 OPENAI_API_KEY 를 자동으로 읽는다
client = OpenAI()
print(type(client).__name__)

OpenAI


## 3. 토큰 — 요금과 길이의 단위 (교재 p.8~9)

- 모델은 글자를 **토큰(서브워드)** 으로 잘라서 읽고 쓴다. 요금도, 길이 제한도 토큰 단위
- 자주 쓰는 단어는 통째로 1토큰, 드문 단어는 여러 조각

`tiktoken` 으로 직접 잘라 본다. (`o200k_base` 는 GPT-4o 계열부터 쓰는 토큰 사전 — gpt-5-nano 의 실제 사용량과 거의 맞는지는 4절에서 확인)

In [6]:
import tiktoken
enc = tiktoken.get_encoding('o200k_base')       # 토큰 사전

def show_tokens(text):
    """text 를 토큰으로 자른 조각과 개수를 보여 준다."""
    ids = enc.encode(text)                                                    # 글자 → 토큰 번호
    pieces = [enc.decode_single_token_bytes(i).decode('utf-8', errors='replace') for i in ids]   # 번호 → 조각
    print(f'{len(ids):>2}토큰  {text!r}  →  {pieces}')
    return len(ids)

# 교재 p.9 의 예
for t in ['I am a', 'Chatbot', '먹었다', '대규모 언어 모델']:
    show_tokens(t)

 3토큰  'I am a'  →  ['I', ' am', ' a']
 2토큰  'Chatbot'  →  ['Chat', 'bot']
 2토큰  '먹었다'  →  ['먹', '었다']
 6토큰  '대규모 언어 모델'  →  ['대', '규', '모', ' 언', '어', ' 모델']


- `I am a` 는 단어마다 1토큰, `Chatbot` → `Chat` + `bot` (교재와 같음)
- `먹었다` 는 교재의 `먹 + 었 + 다`(3개)가 아니라 **`먹` + `었다` 2개**로 잘렸다. 교재는 '서브워드로 쪼갠다' 는 개념 예시이고, 실제 조각은 모델의 토큰 사전마다 다르다
- 한국어는 영어보다 토큰이 많이 든다 — `대규모 언어 모델` 8글자가 6토큰

수업에서 보낸 두 프롬프트의 토큰 수 :

In [7]:
system_prompt = '너는 친절하고 정확한 답변을 하는 AI 비서야'
user_prompt = '최대 200 토큰 내에서 대규모 언어 모델에 대해서 알려줘'

n_system = show_tokens(system_prompt)
n_user = show_tokens(user_prompt)
print('두 프롬프트 합계 :', n_system + n_user, '토큰')

15토큰  '너는 친절하고 정확한 답변을 하는 AI 비서야'  →  ['너', '는', ' 친', '절', '하고', ' 정확', '한', ' 답', '변', '을', ' 하는', ' AI', ' 비', '서', '야']
18토큰  '최대 200 토큰 내에서 대규모 언어 모델에 대해서 알려줘'  →  ['최', '대', ' ', '200', ' 토', '큰', ' 내', '에서', ' 대', '규', '모', ' 언', '어', ' 모델', '에', ' 대해서', ' 알려', '줘']
두 프롬프트 합계 : 33 토큰


## 4. Chat Completions API — 질문 하나에 답 하나

`messages` 는 **역할(role)** 이 붙은 메시지 목록이다.

| role | 뜻 |
|---|---|
| `system` | 모델의 역할 · 말투 · 규칙 (시스템 프롬프트) |
| `user` | 사용자의 요청 (사용자 프롬프트) |
| `assistant` | 모델이 한 답 (대화를 이어 갈 때 다시 넣어 준다) |

In [8]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)
# Chat Completions API 호출
response = client.chat.completions.create(
    model='gpt-5-nano',
    messages=[
        {'role': 'system', 'content': system_prompt},   # 시스템 프롬프트 : 모델에게 주는 역할
        {'role': 'user', 'content': user_prompt},       # 사용자 프롬프트 : 실제 질문
    ],
)
print(f'응답 확인: \n{response}')

응답 확인: 
ChatCompletion(id='chatcmpl-EjF2ARDkvOyhWGGeVFl0AgMVLaqTF', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='대규모 언어 모델(LLM)은 대량의 텍스트를 학습해 입력에 맞춘 텍스트를 생성하거나 이해하는 AI입니다. 주로 트랜스포머 아키텍처를 쓰며, 어텐션으로 긴 문맥을 효과적으로 처리합니다. 학습은 보통 다음 토큰 예측으로 이루어지며, 필요 시 사람의 피드백을 반영하는 RLHF(강화학습 기반 조정)로 성능과 안전성을 개선합니다. 프롬프트에 따라 대화, 요약, 번역, 코딩 보조 등 다양한 작업이 가능하죠. 장점은 강력한 자연어 생성, 넓은 적용 범위, 도메인 지식 활용 능력입니다. 단점으로는 최신 정보 반영의 한계, 환각(잘못된 정보 생성)과 편향의 가능성, 높은 계산 자원과 에너지 소모, 안전/윤리 이슈가 있습니다. 응용 예로 챗봇, 문서 요약, 코드 작성 등이 있습니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1794451258, model='gpt-5-nano-2025-08-07', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=1519, prompt_tokens=43, total_tokens=1562, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_token

응답은 글자가 아니라 **`ChatCompletion` 객체**다. 필요한 값은 점(`.`)과 번호로 꺼낸다.

```
response
 ├─ choices[0]              ← 답 후보 목록의 첫 번째
 │   ├─ message.content     ← ★ 모델이 쓴 글
 │   └─ finish_reason       ← 'stop'(다 썼음) / 'length'(토큰 한도에 걸려 잘림)
 ├─ model                   ← 실제로 답한 모델 버전
 └─ usage                   ← 토큰 사용량 (요금)
```

In [9]:
# 생성된 답변(텍스트)만 추출
response_text = response.choices[0].message.content
print(f'AI 답변: \n{response_text}')
print()
print('끝난 이유 :', response.choices[0].finish_reason, '| 모델 :', response.model)

AI 답변: 
대규모 언어 모델(LLM)은 대량의 텍스트를 학습해 입력에 맞춘 텍스트를 생성하거나 이해하는 AI입니다. 주로 트랜스포머 아키텍처를 쓰며, 어텐션으로 긴 문맥을 효과적으로 처리합니다. 학습은 보통 다음 토큰 예측으로 이루어지며, 필요 시 사람의 피드백을 반영하는 RLHF(강화학습 기반 조정)로 성능과 안전성을 개선합니다. 프롬프트에 따라 대화, 요약, 번역, 코딩 보조 등 다양한 작업이 가능하죠. 장점은 강력한 자연어 생성, 넓은 적용 범위, 도메인 지식 활용 능력입니다. 단점으로는 최신 정보 반영의 한계, 환각(잘못된 정보 생성)과 편향의 가능성, 높은 계산 자원과 에너지 소모, 안전/윤리 이슈가 있습니다. 응용 예로 챗봇, 문서 요약, 코드 작성 등이 있습니다.

끝난 이유 : stop | 모델 : gpt-5-nano-2025-08-07


### 4-1. 사용량 뜯어보기 — "최대 200 토큰" 이라고 썼는데?

In [10]:
usage = response.usage
reasoning = usage.completion_tokens_details.reasoning_tokens      # 답을 쓰기 전에 '생각' 하는 데 쓴 토큰 (화면에 안 보임)
visible = usage.completion_tokens - reasoning                     # 실제로 보이는 답의 토큰

print(f'입력(prompt) 토큰      : {usage.prompt_tokens:>5}   ← 3절에서 센 {n_system + n_user} + 역할 표시 등 포장 {usage.prompt_tokens - n_system - n_user}')
print(f'출력(completion) 토큰  : {usage.completion_tokens:>5}')
print(f'  ├ 추론(reasoning)    : {reasoning:>5}   ({reasoning / usage.completion_tokens:.0%})')
print(f'  └ 보이는 답          : {visible:>5}   ← tiktoken 으로 답을 직접 세면 {len(enc.encode(response_text))}')
print(f'합계                   : {usage.total_tokens:>5}')

입력(prompt) 토큰      :    43   ← 3절에서 센 33 + 역할 표시 등 포장 10
출력(completion) 토큰  :  1519
  ├ 추론(reasoning)    :  1280   (84%)
  └ 보이는 답          :   239   ← tiktoken 으로 답을 직접 세면 230
합계                   :  1562


- 프롬프트에 **"최대 200 토큰 내에서"** 라고 썼지만 보이는 답만 239토큰, 전체 출력은 **1,519토큰**이었다
- 출력의 84% 가 **추론 토큰** — `gpt-5` 계열은 답하기 전에 속으로 생각하는 추론 모델이고, 그 토큰도 **출력 요금**으로 계산된다
- 프롬프트의 "200 토큰" 은 **부탁**일 뿐이다. 모델은 토큰 수를 정확히 세지 못한다. 진짜 상한은 API 매개변수로 건다 (4-3)

![토큰 사용량 애니메이션](./images/anim_02_token_usage.gif)

In [11]:
# 요금 계산 — 1M(100만) 토큰당 USD (입력, 출력). 2026-11 확인 값이며 바뀔 수 있으니 공식 요금표를 본다
PRICE_PER_1M = {'gpt-5-nano': (0.05, 0.40)}

def cost_usd(input_tokens, output_tokens, model='gpt-5-nano'):
    """토큰 수 → 요금(USD)"""
    p_in, p_out = PRICE_PER_1M[model]
    return input_tokens * p_in / 1_000_000 + output_tokens * p_out / 1_000_000

c = cost_usd(usage.prompt_tokens, usage.completion_tokens)
print(f'이번 호출 1번 : ${c:.6f}   → 1,000번이면 ${c * 1000:.2f}')
print(f'출력이 차지하는 비중 : {cost_usd(0, usage.completion_tokens) / c:.1%}')

이번 호출 1번 : $0.000610   → 1,000번이면 $0.61
출력이 차지하는 비중 : 99.6%


출력 토큰이 입력보다 8배 비싸고 양도 많아서, 요금의 대부분은 **출력(그중에서도 추론)** 이다.

### 4-2. 내 필기의 실행 — 시스템 프롬프트가 '날씨 안내 AI'

필기에서는 시스템 프롬프트를 `'너는 친절하고 정확한 날씨 안내를 해주는 AI야'` 로 두고 같은 질문(대규모 언어 모델)을 했다.

In [12]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)
my_response = client.chat.completions.create(
    model='gpt-5-nano',
    messages=[
        {'role': 'system', 'content': '너는 친절하고 정확한 날씨 안내를 해주는 AI야'},
        {'role': 'user', 'content': user_prompt},
    ],
)
my_text = my_response.choices[0].message.content
print(f'AI 답변:\n {my_text}')

AI 답변:
 대규모 언어 모델(LLM)은 방대한 텍스트를 바탕으로 다음 단어를 예측하도록 학습한 인공지능이다. 트랜스포머 계열 아키텍처를 주로 쓰며 수십억에서 수조 개의 매개변수를 가진다. 학습은 대규모 데이터 코퍼스로의 자기회귀 학습과 필요 시 강화학습 기반의 정렬(RLHF)로 미세조정된다. 주요 능력은 자연어 이해·생성, 대화, 번역, 요약, 코딩 보조 등이다. 맥락 파악으로 긴 글을 작성하고 질의에 응답할 수 있지만, 학습 데이터의 편향이나 부정확한 정보를 반영할 수 있고 때로는 망상처럼 보이는 응답을 낼 수 있다. 또한 높은 계산 자원과 에너지가 필요하고 안전·프라이버시 이슈가 있다. 활용 예로는 챗봇, 문서 작성 도구, 고객지원, 코드 보조 등이 있다. 필요하면 더 자세히 설명해줄게.


In [13]:
# 두 답 비교 — 글자 수와 토큰 수
for name, text in [('AI 비서 (강의 코드)', response_text), ('날씨 안내 AI (내 필기)', my_text)]:
    print(f'{name:<18} {len(text):>4}자  {len(enc.encode(text)):>4}토큰  끝맺음: …{text[-18:]}')

AI 비서 (강의 코드)       376자   230토큰  끝맺음: …요약, 코드 작성 등이 있습니다.
날씨 안내 AI (내 필기)     385자   239토큰  끝맺음: … 필요하면 더 자세히 설명해줄게.


- 역할이 '날씨 안내' 여도 질문이 LLM 이면 LLM 을 설명한다 — 시스템 프롬프트는 **질문과 맞을 때** 효과가 있다. 날씨만 답하게 하려면 "날씨 외의 질문은 정중히 거절해" 같은 **규칙**까지 써야 한다
- 말투는 달라졌다 : 한쪽은 `~입니다`, 다른 쪽은 `~이다 … 설명해줄게`
- 두 답 모두 200토큰을 넘겼다 → "200 토큰 내" 는 지켜지지 않는다

> **📝 고친 부분** — 필기는 LLM 을 묻는 실습에 날씨 안내 역할을 줬다(다른 예제에서 가져온 문장). 역할과 질문을 맞췄고, 위처럼 비교용으로만 남겼다.

### 4-3. 길이를 진짜로 제한하려면

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
# 출력 토큰 상한을 매개변수로 건다 (gpt-5 계열은 max_tokens 가 아니라 max_completion_tokens)
limited = client.chat.completions.create(
    model='gpt-5-nano',
    messages=[
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': '대규모 언어 모델을 세 문장으로 설명해줘'},   # 토큰 수 대신 '문장 수' 로 부탁
    ],
    reasoning_effort='minimal',        # 추론을 최소로 → 추론 토큰이 줄어든다
    max_completion_tokens=400,         # 추론 + 보이는 답을 합친 상한
)
print(limited.choices[0].message.content)
print('끝난 이유 :', limited.choices[0].finish_reason)   # 'length' 면 상한에 걸려 잘린 것
print(limited.usage)

직접 확인할 것 : `max_completion_tokens` 는 **추론 토큰까지 포함**한 상한이라, 너무 작게 잡으면 생각만 하다 끝나서 답이 빈 문자열(`finish_reason='length'`)이 될 수 있다. 길이는 '세 문장' · '300자' 처럼 **사람이 세는 단위**로 부탁하는 편이 잘 지켜진다.

## 5. Responses API — 웹 검색 도구 붙이기

Chat Completions 는 모델이 **학습한 것**만으로 답한다. '오늘 뉴스' 는 모른다. Responses API 는 `tools` 에 **웹 검색**을 넣으면 모델이 스스로 검색하고 그 결과로 답한다.

| | Chat Completions | Responses |
|---|---|---|
| 호출 | `client.chat.completions.create` | `client.responses.create` |
| 시스템 프롬프트 | `messages` 안의 `role: system` | **`instructions=`** |
| 사용자 프롬프트 | `messages` 안의 `role: user` | **`input=`** |
| 답 꺼내기 | `response.choices[0].message.content` | **`response.output_text`** |
| 내장 도구 | 없음 | `tools=[{"type": "web_search_preview"}]` 등 |
| 사용량 | `usage.prompt_tokens` · `completion_tokens` | `usage.input_tokens` · `output_tokens` |

In [14]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)
# Responses API 호출 — 웹 검색 실행
web_response = client.responses.create(
    model='gpt-5-nano',
    tools=[{'type': 'web_search_preview'}],          # 웹 검색 도구 활성화
    instructions='너는 리서치 전문가이며, 사용자 요청에 대해서 친절하고 정확한 답변을 해야 해',   # 시스템 프롬프트
    input='오늘 주요 글로벌 IT 및 AI 관련 핵심 뉴스를 3가지로 요약해 줘',                       # 사용자 프롬프트
)
print(web_response.output_text)                      # 답 글자만

오늘(2026년 11월 12일 기준) 글로벌 IT/AI 분야의 핵심 이슈 3건을 요약합니다.

- CoreWeave의 분기 실적 발표로 AI 인프라 주가 급등
  - 요지: 네오클라우드(neocloud) 플랫폼인 CoreWeave가 3분기 실적을 발표하며 AI 인프라 수요에 대한 우려를 다소 완화했고, 이에 따라 주요 AI 관련 주가가 반등했습니다. CoreWeave의 주가가 큰 폭으로 상승했고 Nvidia도 반등하는 흐름이 나타났습니다. ([marketscreener.com](https://www.marketscreener.com/news/stocks-gain-as-coreweave-reignites-ai-trade-ce785edbda80f726?utm_source=openai))

- OpenAI, 데이터센터 정책 프레임워크 발표로 지역사회 신뢰 회복 추진
  - 요지: 중간선거 직후 데이터센터 운영을 둘러싼 비판이 커지는 가운데 OpenAI가 데이터센터 deals에서 NDA를 지양하고 주민이 전기 비용 부담을 지지 않도록 하는 등 지역사회 신뢰를 키우려는 새로운 정책 프레임워크를 공개했습니다. 이 변화는 AI 데이터센터 확산에 대한 규제 환경과 기업의 책임 논의에 중요한 계기가 되고 있습니다. ([bloomberg.com](https://bloomberg.com/news/articles/2026-11-05/openai-to-change-how-it-handles-data-center-efforts-after-outcry?utm_source=openai))

- 유럽에서 데이터센터 반대 시위 확산, 정책 논쟁 지속
  - 요지: AI 데이터센터 확산에 반대하는 시위가 유럽 여러 도시에서 이어졌고, 중간선거의 정치적 맥락 속에서도 데이터센터 투자와 규제에 대한 논의가 계속되고 있습니다. 해당 현상은 지역사회 영향과 에너지 인프라 이슈를 둘러싼 국제적 관심을 불러일으키고 있습니다. ([lemonde.fr](https://www.lemonde.fr/en/econ

답에 **오늘 날짜**와 **출처 링크**가 붙었다. `output` 에는 답만 있는 게 아니라 모델이 거친 단계가 순서대로 들어 있다.

In [15]:
from collections import Counter

# output 에 담긴 항목의 종류
types = [item.type for item in web_response.output]
print('항목 수 :', len(types), dict(Counter(types)))
print('순서    :', ' → '.join(types[:6]), '→ …', '→', types[-1])
print()

# 모델이 스스로 만든 검색어
n = 0
for item in web_response.output:
    if item.type == 'web_search_call':
        n += 1
        print(f'검색 {n} : {item.action.queries}')

항목 수 : 18 {'reasoning': 9, 'web_search_call': 8, 'message': 1}
순서    : reasoning → web_search_call → reasoning → web_search_call → reasoning → web_search_call → … → message

검색 1 : ['November 12, 2026 IT AI news today', 'global IT AI news November 2026 Reuters', 'Tech News November 12 2026 AI']
검색 2 : ['Reuters tech AI November 12, 2026', 'Bloomberg technology AI November 12 2026', 'CNBC AI November 12 2026']
검색 3 : ['Reuters technology November 12, 2026 AI', 'Reuters November 12, 2026 AI news CoreWeave November 12 2026']
검색 4 : ['Reuters CoreWeave results Nov 2026 AI spending AI cloud stocks rally', 'Reuters CoreWeave beats quarterly revenue November 2026']
검색 5 : ['Reuters OpenAI DevDay 2026 November', 'Bloomberg OpenAI 2026 DevDay November']
검색 6 : ['Reuters Nvidia November 12, 2026 AI stock rally', 'Reuters AI news November 12 2026 CoreWeave Nvidia']
검색 7 : ['Reuters OpenAI data centers changes November 2026', 'OpenAI data centers outcry November 2026 Reuters']
검색 8 : ['Reuters Cor

'오늘 뉴스 3가지' 한 줄에 모델이 **8번 검색**했다. 처음에는 넓게(`IT AI news today`), 뒤로 갈수록 찾은 기사를 확인하는 검색(`CoreWeave results`, `OpenAI data centers`)으로 좁혀 간다 — 생각 → 검색 → 생각 → … → 답. 이것이 **에이전트**의 기본 모양이다.

![웹 검색 과정 애니메이션](./images/anim_03_web_search_loop.gif)

In [16]:
# 웹 검색 호출의 사용량과 요금
wu = web_response.usage
print(f'입력 {wu.input_tokens:,} · 출력 {wu.output_tokens:,} (추론 {wu.output_tokens_details.reasoning_tokens:,}) · 합계 {wu.total_tokens:,} 토큰')

web_cost = cost_usd(wu.input_tokens, wu.output_tokens)
chat_cost = cost_usd(usage.prompt_tokens, usage.completion_tokens)
print(f'토큰 요금 : ${web_cost:.5f}  (4절 일반 호출 ${chat_cost:.5f} 의 {web_cost / chat_cost:.1f}배)')
print(f'입력 토큰은 {wu.input_tokens / usage.prompt_tokens:,.0f}배 — 검색해 온 웹 페이지 글이 전부 입력으로 들어가기 때문')

입력 40,764 · 출력 5,157 (추론 4,416) · 합계 45,921 토큰
토큰 요금 : $0.00410  (4절 일반 호출 $0.00061 의 6.7배)
입력 토큰은 948배 — 검색해 온 웹 페이지 글이 전부 입력으로 들어가기 때문


질문은 한 줄인데 입력이 4만 토큰이다. 검색 결과(웹 페이지 본문)가 모델에게 **입력으로** 들어가기 때문이다. 여기에 웹 검색 도구 호출 요금이 따로 붙는다(공식 요금표 참고).

### 5-1. 내 필기의 오류 — `inputs=`

In [17]:
# 필기 원본 그대로 : input 을 inputs 로 적었다 → 서버에 가기 전에 파이썬에서 막힌다
try:
    client.responses.create(
        model='gpt-5-nano',
        tools=[{'type': 'web_search_preview'}],
        instructions='너는 리서치 전문가야, 사용자 요청에 대해서 친절한 답변해',
        inputs='hot 뉴스 요약해서 알려줘',
    )
except TypeError as e:
    print('TypeError :', e)

TypeError : Responses.create() got an unexpected keyword argument 'inputs'. Did you mean 'input'?


> **📝 고친 부분** — `inputs=` → **`input=`**. 매개변수 이름이 틀리면 요청을 보내기도 전에 `TypeError` 가 난다(요금 없음). 오류 메시지 끝의 `Did you mean 'input'?` 을 읽으면 된다. 프롬프트도 `hot 뉴스` 보다 '무엇을 · 몇 개 · 어떤 분야' 를 적은 강의 코드 쪽이 검색어가 구체적으로 나온다.

### 5-2. 주소를 주고 읽게 하기

In [18]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)
# 사용자 프롬프트에 주소를 넣으면 그 페이지를 열어 읽는다
prompt = """
https://platform.openai.com/docs/api-reference/responses/create
를 읽어서 Responses API에 대해 300 토큰 이내에서 요약 정리해주세요.
"""
url_response = client.responses.create(
    model='gpt-5-nano',
    tools=[{'type': 'web_search_preview'}],
    instructions='리서치 전문가이며, 사용자 요청에 대해서 친절하고 정확한 답변을 해야 해',
    input=prompt,
)
print(f'응답 확인\n{url_response.output_text}')

응답 확인
Responses API는 입력(텍스트/이미지 등)을 바탕으로 모델의 응답을 생성하고 이를 관리하는 플랫폼 API입니다. 텍스트 입력, 이미지 입력, 파일 입력 등 다양한 입력 타입을 지원하고, 모델이 텍스트나 JSON 출력을 생성하도록 합니다. 또한 사용자가 자신의 코드나 내장 도구(웹 검색, 파일 검색 등)를 호출해 입력 데이터를 활용해 응답을 구성하도록 할 수 있습니다. 주요 엔드포인트로는 POST /responses(모델 응답 생성), Retrieve/Delete(응답 조회/삭제), List input items, Count input tokens, Cancel a response, Compact a response가 있습니다. 또한 Beta API로 “Responses (Multi-agent)” 버전도 있어 Create/Retrieve/Delete 등의 작업이 가능하며, 입력 아이템 관리와 토큰 카운트 같은 보조 기능도 문서에 포함되어 있습니다. ([developers.openai.com](https://developers.openai.com/api/reference/resources/responses/methods/create))




- 준 주소는 `platform.openai.com/docs/…` 인데 답의 출처는 `developers.openai.com/api/reference/…` 다 — 문서가 옮겨져 있어 모델이 바뀐 주소를 따라갔다
- 웹 검색을 붙여도 요약이 **맞는지는 내가 확인**해야 한다. 엔드포인트 이름 같은 세부는 원문과 대조할 것

## 6. 스트리밍 — 만들어지는 대로 받기

지금까지는 답이 **다 만들어진 뒤** 한 번에 왔다(4절 호출은 1,519토큰을 만드는 동안 화면에 아무것도 없다). `stream=True` 를 주면 토큰 조각(chunk)이 만들어지는 대로 온다. ChatGPT 화면에서 글자가 흘러나오는 방식.

![스트리밍 애니메이션](./images/anim_04_streaming.gif)

### 6-1. 필기의 오류 재현 — `'tuple' object has no attribute 'choices'`

필기는 `client.chat.completions.create(..., stream=True)` 의 **결과를 변수에 담지 않았다.** 그래서 아래 `for chunk in response:` 의 `response` 는 **앞 셀의 Responses API 결과**였다.

In [19]:
response = web_response        # 필기 상황 재현 : response 가 직전 웹 검색 응답을 가리킨다

try:
    for chunk in response:                          # 스트림이 아니라 응답 '객체' 를 돌고 있다
        content = chunk.choices[0].delta.content
except AttributeError as e:
    print('AttributeError :', e)

# 객체를 for 로 돌면 무엇이 나오나 — (필드 이름, 값) 튜플
first = next(iter(response))
print(type(first).__name__, first)

AttributeError : 'tuple' object has no attribute 'choices'
tuple ('id', 'resp_0751f43462a7536d006af52745445487d2ae5b2ba75d389079')


응답 객체(pydantic 모델)를 `for` 로 돌면 `('id', 'resp_…')` 같은 **(이름, 값) 튜플**이 나온다. 튜플에는 `.choices` 가 없어서 난 오류다. 스트리밍과는 관계없는, **변수에 안 담아서** 생긴 문제.

> **📝 고친 부분**
> - `client.chat.completions.create(...)` → **`stream = client.chat.completions.create(...)`** 로 받아서 그 변수를 돈다
> - 필기의 주석 `# system prompt: instructions 매개변수로 전달` · `# user prompt: input 매개변수로 전달` 은 Responses API 설명이다. Chat Completions 는 둘 다 `messages` 로 전달
> - 마지막 조각은 `content` 가 `None` 이고, 설정에 따라 `choices` 가 빈 조각도 온다 → 둘 다 건너뛴다

### 6-2. 고친 코드

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
# Chat Completions API 호출 — 스트리밍
stream = client.chat.completions.create(
    model='gpt-5-nano',
    messages=[
        {'role': 'system', 'content': system_prompt},                                   # 시스템 프롬프트 (messages 로 전달)
        {'role': 'user', 'content': '최대 100 토큰 내에서 대규모 언어 모델에 대해서 알려줘'},   # 사용자 프롬프트 (messages 로 전달)
    ],
    stream=True,                       # 스트리밍 모드 활성화
)

# 조각(chunk)이 도착하는 대로 이어 붙여 출력
chunks = []
for chunk in stream:
    if not chunk.choices:              # choices 가 빈 조각(사용량 정보 등)은 건너뛴다
        continue
    content = chunk.choices[0].delta.content     # 이번 조각에 새로 온 글자 (message 가 아니라 delta)
    if content:                        # 처음(역할만) · 마지막(None) 조각 제외
        chunks.append(content)
        print(content, end='', flush=True)       # 줄바꿈 없이 바로 화면에

full_text = ''.join(chunks)            # 다 모으면 일반 호출의 message.content 와 같은 글
print(f'\n\n조각 {len(chunks)}개 · {len(full_text)}자')

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)
# 같은 일을 Responses API 로 — 조각이 '이벤트' 로 온다
stream = client.responses.create(
    model='gpt-5-nano',
    instructions=system_prompt,
    input='대규모 언어 모델을 세 문장으로 설명해줘',
    stream=True,
)
for event in stream:
    if event.type == 'response.output_text.delta':    # 글자 조각 이벤트만 골라서
        print(event.delta, end='', flush=True)

| | 일반 응답 | 스트리밍 |
|---|---|---|
| 받는 시점 | 전부 만든 뒤 한 번 | 조각마다 |
| 받는 값 | 응답 객체 1개 | 조각을 내는 스트림 (한 번만 돌 수 있다) |
| 글 꺼내기 | `choices[0].message.content` | `chunk.choices[0].delta.content` 를 이어 붙임 |
| 전체 시간 · 요금 | 같다 | 같다 — **첫 글자가 보이기까지**가 빨라질 뿐 |
| 쓰는 곳 | JSON 파싱 · 일괄 처리 | 챗봇 화면 |

## 📝 고친 부분 정리

| 위치 | 필기 원본 | 고친 내용 |
|---|---|---|
| 환경 설정 | `key = "C:\\Users\\Jinho\\work\\.env"` 절대 경로 | `find_dotenv()` 로 찾기 · 키가 있는지만 확인 |
| Chat Completions | LLM 질문에 시스템 프롬프트 '날씨 안내 AI' | 질문에 맞는 역할 · 날씨 버전은 비교용으로 |
| Chat Completions | "최대 200 토큰" 이 지켜진다고 가정 | 실제 사용량 확인 — 보이는 답 239 · 추론 1,280 · 합계 1,519. 상한은 `max_completion_tokens` |
| Web Search | `inputs="hot 뉴스 …"` → `TypeError` | `input=` |
| Web Search | `print(response)` 로 7만 자 출력 | `output_text` · 검색어 · 사용량만 골라 출력 |
| 스트리밍 | `create(...)` 결과를 변수에 안 담음 → 앞 셀의 `response` 를 돌아 `'tuple' object has no attribute 'choices'` | `stream = …` 로 받아서 순회 |
| 스트리밍 | 반복문이 `"""` 문자열 안에 들어 있음(실행 안 됨) | 코드로 꺼냄 · 빈 조각 건너뛰기 |
| 스트리밍 주석 | `instructions` · `input` 매개변수로 전달 | Chat Completions 는 `messages` 로 전달 |
| 제목 | `#Chatbot API 실습` (`#` 뒤 공백 없음 → 제목으로 안 보임) · `comletion` · `새성된` · `요쳥` · `챗팅` | `# ` 띄어쓰기 · completion · 생성된 · 요청 · 채팅 |
| 교재 p.9 | `먹었다 → 먹 + 었 + 다` | 개념 예시. 실제 `o200k_base` 는 `먹` + `었다` 2토큰 |
| 교재 p.7 | API 키 발급 '결재 후' | 결제 |
| 교재 머리글 | `SK네트웍스 Famiy` | Family |